In [4]:
import os
import torch
import math
import time
import sys
import inspect
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from dataclasses import dataclass
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau

from tqdm.notebook import tqdm
sys.path.append('..')
from model import Transformer, load_model, save_model

In [2]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
device_type = "cuda" if device.type.startswith("cuda") else "cpu"

In [ ]:
@dataclass
class CFG:
    train_data : str ='../data/tokenized_train.bin'
    valid_data : str ='../data/tokenized_valid.bin'
    experiment_name: str = "tiny_stories"
    max_seq_len: int = 512
    vocab_size: int = 16000
    dim: int = 512
    n_layers: int = 8
    n_heads_q: int = 8
    n_heads_kv: Optional[int] = 4
    dropout: float = 0.0  #no dropout as in the original paper

    multiple_of: int = 256
    ffn_multiplier: Optional[int] = None
    norm_eps: float = 1e-6
    init_std: float = 0.02

    state_dir: str = "model_states_tiny_stories"
    grad_clip: float = 1.0
    grad_accumulation_steps: int = 1
    batch_size_train: int = 64
    batch_size_valid: int = 64
    num_workers: int = 8
    n_epochs: int = 2
    last_state: Optional[str] = None
    best_state_file_name: str = "best_model"
    last_state_file_name: str = "last_model"

    init_lr: float = 1e-3
    min_lr: float = 1e-4
    warmup_steps: int = 200
    max_steps: int = 32500
    valid_every_steps: int = 1000
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)

args = CFG()

In [4]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self,data_path: str, block_size: int):
        self.data = np.memmap(data_path, dtype=np.uint16)
        self.block_size = block_size

    def __len__(self):
        return len(self.data)//self.block_size-1

    def __getitem__(self, idx):
        x = self.data[idx*self.block_size:(idx+1)*self.block_size]
        y = self.data[idx*self.block_size+1:(idx+1)*self.block_size+1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [5]:
def lr_at(step: int, cfg: CFG) -> float:
    """Linear warmup for cfg.warmup_steps, then cosine decay to cfg.min_lr."""
    if step < cfg.warmup_steps:
        return cfg.init_lr*(step+1)/cfg.warmup_steps
    if step >= cfg.max_steps:
        return cfg.min_lr
    progress = (step - cfg.warmup_steps)/max(1, cfg.max_steps - cfg.warmup_steps)
    return cfg.min_lr + 0.5*(cfg.init_lr - cfg.min_lr)*(1.0 + math.cos(math.pi*progress))


In [ ]:
def train(
    train_config: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    train_loader: torch.utils.data.DataLoader,
    valid_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    loss_fn: torch.nn.modules.loss.CrossEntropyLoss,
    vocab_size: int,
    device: torch.device,
    summary_writer: SummaryWriter,
    global_step: int,
    best_loss: float
    ):

    batch_loss = np.array([])
    norms = np.array([])
    model.train()
    batch_iterator = tqdm(train_loader, desc=f"Processing Epoch {epoch_idx}")
    t0 = time.time()
    tokens_processed = 0
    for step,(x, y) in enumerate(batch_iterator):
        pos = torch.arange(x.shape[1], device=device).repeat(x.shape[0], 1)
        inp = x.to(device, non_blocking=True)
        labels = y.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            out = model(inp, pos, mask=None)
            loss = loss_fn(out.view(-1, vocab_size),labels.view(-1))

        batch_loss = np.append(batch_loss, [loss.item()])
        tokens_processed += inp.numel()
        loss = loss / train_config.grad_accumulation_steps
        loss.backward()

        if (step+1)%train_config.grad_accumulation_steps == 0:
            # warmup + cosine, applied per optimizer step
            lr = lr_at(global_step, train_config)
            for group in optimizer.param_groups:
                group['lr'] = lr

            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.grad_clip)
            norms = np.append(norms, [norm.item()])
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            # step-based validation
            if global_step % train_config.valid_every_steps == 0:
                valid_loss = valid(train_config, model, valid_loader, epoch_idx, loss_fn,
                                   vocab_size, device)
                window = batch_loss[-train_config.valid_every_steps:].mean()
                note = ''
                if valid_loss < best_loss:
                    best_loss = valid_loss
                    note = '   <- best, saving'
                    save_model(model, optimizer, global_step, device=device,
                               state_file_name=train_config.best_state_file_name,
                               state_dir=train_config.state_dir, loss=best_loss)
                print('step {:>6}  lr {:.2e}  train {:.4f}  valid {:.4f}{}'.format(
                    global_step, lr, window, valid_loss, note))
                summary_writer.add_scalar('Loss/Valid', valid_loss, global_step)
                summary_writer.add_scalar('Loss/TrainWindow', window, global_step)
                summary_writer.add_scalar('Learning Rate', lr, global_step)

            if global_step >= train_config.max_steps:
                break

        batch_iterator.set_postfix({"loss": f"{loss.item()*train_config.grad_accumulation_steps:.4f}"})

    dt = time.time() - t0
    tokens_per_sec = tokens_p rocessed/dt
    epoch_loss = batch_loss.mean()
    epoch_norm = norms.mean() if len(norms) else float('nan')
    print('Epoch: {} Train Loss: {:.3f} Average Norm: {:.3f} Tokens/sec: {:.2f} Time: {:.2f}s'.format(
        epoch_idx, epoch_loss, epoch_norm, tokens_per_sec, dt))
    return epoch_loss, global_step, best_loss


In [7]:
def valid(
    train_config: CFG,
    model: Transformer,
    valid_loader: torch.utils.data.DataLoader,
    epoch_idx: int,
    loss_fn: torch.nn.modules.loss.CrossEntropyLoss,
    vocab_size: int,
    device: torch.device,
    max_batches: Optional[int] = None
    )-> float:

    batch_loss = np.array([])
    model.eval()
    batch_iterator = valid_loader
    with torch.no_grad():
        for step,(x, y) in enumerate(batch_iterator):
            pos = torch.arange(x.shape[1], device=device).repeat(x.shape[0], 1)
            inp = x.to(device)
            labels = y.to(device)

            with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                out = model(inp, pos, mask=None)
                loss = loss_fn(out.view(-1, vocab_size),labels.view(-1))

            batch_loss = np.append(batch_loss, [loss.item()])
            if max_batches is not None and step+1 >= max_batches:
                break
    epoch_loss = batch_loss.mean()
    return epoch_loss

In [8]:
def main(
    args: CFG,
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    dataset_train: torch.utils.data.Dataset,
    dataset_valid: torch.utils.data.Dataset,
    scheduler,                      # unused: the cosine schedule is applied per step inside train()
    summary_writer: SummaryWriter,
    device: torch.device
    ):

    train_loader = DataLoader(dataset_train, batch_size=args.batch_size_train,
                              num_workers=args.num_workers, shuffle=True,
                              drop_last=True, pin_memory=True)
    valid_loader = DataLoader(dataset_valid, batch_size=args.batch_size_valid,
                              num_workers=2, pin_memory=True)
    loss_fn = nn.CrossEntropyLoss()
    vocab_size = args.vocab_size
    model.to(device)

    best_loss = float(np.inf)
    global_step = 0
    if args.last_state is not None:
        state_path = os.path.join(args.state_dir, args.last_state)
        state = torch.load(state_path)
        model._orig_mod.load_state_dict(state['model_state_dict'])
        optimizer.load_state_dict(state['optimizer_state_dict'])
        best_loss = state['loss'].item()
        global_step = state['step'].item()
        print(f"Loaded model from {args.last_state}")
        print(f"Best Loss: {best_loss:.4f}, Resuming from step {global_step}")

    print('warmup {} steps, cosine {:.1e} -> {:.1e} over {} steps'.format(
        args.warmup_steps, args.init_lr, args.min_lr, args.max_steps))
    print('validating every {} steps on {} batches\n'.format(
        args.valid_every_steps, args.valid_batches if hasattr(args, 'valid_batches') else str(len(valid_loader))))

    t_begin = time.time()
    for epoch in range(args.n_epochs):
        train_loss, global_step, best_loss = train(
            args, model, optimizer, train_loader, valid_loader, epoch, loss_fn,
            vocab_size, device, summary_writer, global_step, best_loss)
        summary_writer.add_scalar('Loss/Train', train_loss, epoch)
        if global_step >= args.max_steps:
            print('Reached max_steps ({}).'.format(args.max_steps))
            break

    save_model(model, optimizer, global_step, device=device,
               state_file_name=args.last_state_file_name,
               state_dir=args.state_dir, loss=best_loss)
    print("Total time: {:.2f}, Best Valid Loss: {:.3f}".format(time.time() - t_begin, best_loss))


In [9]:
d_train = Dataset(args.train_data, args.max_seq_len)
d_valid = Dataset(args.valid_data, args.max_seq_len)

In [10]:
model = Transformer(args)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params/1e6:.2f}M")

Trainable parameters: 41.56M


In [11]:
def get_optimizer_params(model, lr, weight_decay=0.0):
    no_decay = ["bias", "norm"]
    optimizer_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'lr': lr, 'weight_decay': 0.0}
    ]
    return optimizer_parameters


In [12]:
optimizer_params = get_optimizer_params(model, args.init_lr, args.weight_decay)
fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
use_fused = fused_available and device_type == "cuda" 

optimizer = torch.optim.AdamW(
    optimizer_params,
    lr = args.init_lr,
    betas=args.betas,
    fused=use_fused
)

schedular = ReduceLROnPlateau(optimizer, mode = 'min', factor = 0.3, patience = 2, min_lr= 1e-6)

In [13]:
model.to(device)
model = torch.compile(model)

In [14]:
summary_writer = SummaryWriter(log_dir=args.experiment_name)

In [15]:
%%time
main(args, model, optimizer, d_train, d_valid, schedular, summary_writer, device)

warmup 200 steps, cosine 1.0e-03 -> 1.0e-04 over 32500 steps
validating every 1000 steps on 165 batches



Processing Epoch 0:   0%|          | 0/16289 [00:00<?, ?it/s]

W1019 08:46:41.626000 58485 site-packages/torch/_inductor/utils.py:1806] [0/0] Not enough SMs to use max_autotune_gemm mode


step   1000  lr 9.99e-04  train 2.3920  valid 1.7114   <- best, saving
step   2000  lr 9.93e-04  train 1.6226  valid 1.5534   <- best, saving
step   3000  lr 9.83e-04  train 1.5147  valid 1.4796   <- best, saving
step   4000  lr 9.70e-04  train 1.4562  valid 1.4317   <- best, saving
step   5000  lr 9.52e-04  train 1.4166  valid 1.3989   <- best, saving
step   6000  lr 9.30e-04  train 1.3887  valid 1.3722   <- best, saving
step   7000  lr 9.05e-04  train 1.3663  valid 1.3532   <- best, saving
step   8000  lr 8.77e-04  train 1.3491  valid 1.3351   <- best, saving
step   9000  lr 8.45e-04  train 1.3306  valid 1.3217   <- best, saving
step  10000  lr 8.11e-04  train 1.3181  valid 1.3086   <- best, saving
step  11000  lr 7.74e-04  train 1.3033  valid 1.2950   <- best, saving
step  12000  lr 7.35e-04  train 1.2920  valid 1.2846   <- best, saving
step  13000  lr 6.94e-04  train 1.2810  valid 1.2735   <- best, saving
step  14000  lr 6.52e-04  train 1.2700  valid 1.2628   <- best, saving
step  

Processing Epoch 1:   0%|          | 0/16289 [00:00<?, ?it/s]

step  17000  lr 5.22e-04  train 1.2260  valid 1.2345   <- best, saving
step  18000  lr 4.78e-04  train 1.2200  valid 1.2249   <- best, saving
step  19000  lr 4.35e-04  train 1.2151  valid 1.2164   <- best, saving
step  20000  lr 3.94e-04  train 1.2061  valid 1.2076   <- best, saving
step  21000  lr 3.53e-04  train 1.1970  valid 1.1993   <- best, saving
step  22000  lr 3.15e-04  train 1.1900  valid 1.1904   <- best, saving
step  23000  lr 2.79e-04  train 1.1823  valid 1.1822   <- best, saving
step  24000  lr 2.45e-04  train 1.1730  valid 1.1749   <- best, saving
step  25000  lr 2.15e-04  train 1.1666  valid 1.1676   <- best, saving
step  26000  lr 1.87e-04  train 1.1583  valid 1.1606   <- best, saving
step  27000  lr 1.63e-04  train 1.1530  valid 1.1545   <- best, saving
step  28000  lr 1.42e-04  train 1.1472  valid 1.1490   <- best, saving
step  29000  lr 1.26e-04  train 1.1420  valid 1.1442   <- best, saving
step  30000  lr 1.13e-04  train 1.1385  valid 1.1400   <- best, saving
step  